## Test Agent Calling (agent_call) - Multi-Agent Delegation

This notebook demonstrates **agent_call** - the ability for an LLM to delegate work to other agents.

When an agent's LLM produces an `agent_call` action, Protolink automatically:
1. Routes the request to the target agent via the transport layer
2. Executes the requested action (tool_call or infer)
3. Returns the result back to the originating LLM's inference loop

### Agent Call Actions

- **`tool_call`**: Execute a specific tool owned by the target agent
- **`infer`**: Ask the target agent's LLM to generate a response

### Setup

We'll create:
1. A **Registry** to enable agent discovery
2. A **Weather Agent** with a `get_weather` tool
3. A **Coordinator Agent** with LLM that delegates weather queries

In [ ]:
from protolink.agents import Agent
from protolink.discovery import Registry
from protolink.llms.api import OpenAILLM  # noqa: F401
from protolink.llms.server import OllamaLLM

# URLs for our agents
REGISTRY_URL = "http://localhost:8000"
WEATHER_AGENT_URL = "http://localhost:8001"
COORDINATOR_URL = "http://localhost:8002"

# Choose your LLM
# llm = OpenAILLM(model="gpt-4o") # should have in env the OPENAI_API_KEY set, otherwise use api_key arg

# Use Ollama and the really light-weight gemma4:latest 8B model I use below to experiment for free
llm = OllamaLLM(base_url="http://localhost:11434", model="gemma4:e4b")

## Start the Registry

The Registry enables agent discovery. Agents register themselves and can discover other agents.

In [ ]:
registry = Registry(url=REGISTRY_URL, transport="http")
registry.start()

## Create and Start the Weather Agent

This agent owns a `get_weather` tool. The Coordinator will delegate weather queries to it.

In [ ]:
weather_agent = Agent(
    card={
        "name": "weather_agent",
        "description": "Provides weather information for any location",
        "url": WEATHER_AGENT_URL,
    },
    transport="http",
    registry=registry,
)


@weather_agent.tool(
    name="get_weather", description="Get current weather for a location", input_schema={"location": str}
)
def get_weather(location: str) -> str:
    """Simulated weather lookup."""
    weather_data = {
        "athens": "Sunny, 28°C",
        "london": "Cloudy, 15°C",
        "tokyo": "Rainy, 22°C",
        "new york": "Partly cloudy, 20°C",
    }
    return weather_data.get(location.lower(), f"Weather data not available for {location}")


weather_agent.start(background=True)
print(f"Weather Agent running at {WEATHER_AGENT_URL}")
print(f"Tools: {list(weather_agent.tools.keys())}")

## Create and Start the Coordinator Agent

This agent has an LLM and knows about the Weather Agent via the Registry. When asked about weather, it will delegate to the Weather Agent using `agent_call`.

In [ ]:
# The following prompt will be added to the existing predefined system prompt given by Protolink.
COORDINATOR_SYSTEM_PROMPT = """You are a coordinator agent. When users ask about weather, delegate to the weather_agent
using agent_call with action 'tool_call'. Use the weather_agent's get_weather tool."""

coordinator = Agent(
    card={
        "name": "coordinator",
        "description": "Coordinates tasks and delegates to specialized agents",
        "url": COORDINATOR_URL,
    },
    transport="http",
    registry=registry,
    llm=llm,
    system_prompt=COORDINATOR_SYSTEM_PROMPT,
)

coordinator.start(background=True)
print(f"Coordinator Agent running at {COORDINATOR_URL}")

## Verify Agent Discovery

Let's confirm the Coordinator can discover the Weather Agent.

In [ ]:
discovered = await coordinator.discover_agents()
print(f"Discovered {len(discovered)} agents:")
for agent in discovered:
    print(f"  - {agent.name}: {agent.description}")
    if agent.skills:
        print(f"    Tools: {[s.id for s in agent.skills]}")

## Send a Task that requires Agent Delegation

We'll ask the Coordinator about weather. It should:
1. Recognize it needs weather data
2. Produce an `agent_call` to the Weather Agent
3. Receive the result and formulate a final response

In [ ]:
from protolink.client import AgentClient
from protolink.models import Task

# Setup client using the Coordinator's transport
client = AgentClient(transport=coordinator.transport)

# Create an infer task
task = Task.create_infer(prompt="What's the weather like in Athens right now?")

print("Sending task to Coordinator...")
result = await client.send_task(agent_url=COORDINATOR_URL, task=task)

print(f"\nResult: {result.get_last_part_content()}")

## What Happened Under the Hood

1. The Coordinator's LLM received the query
2. It recognized this requires the Weather Agent and produced:
```json
{
  "type": "agent_call",
  "action": "tool_call",
  "agent": "weather_agent",
  "tool": "get_weather",
  "args": {"location": "Athens"}
}
```
3. Protolink's `_handle_agent_call` callback sent a Task to the Weather Agent
4. The Weather Agent executed `get_weather("Athens")`
5. The result was returned to the Coordinator's LLM loop
6. The LLM produced a final response incorporating the weather data

## Cleanup

In [ ]:
coordinator.stop()
weather_agent.stop()
registry.stop()
print("All agents stopped.")